# Exercise: LoRA Fine-Tuning for Language Models


> Objectives
>
> 1. Understand Parameter-Efficient Fine-Tuning (PEFT) with LoRA
> 2. Fine-tune a conversational AI model for a specific linguistic task
> 3. Learn how to adapt large language models with minimal resources


## Imports & Setup

- [ ] Ensure you are using the .venv kernel before getting started

<small>Note: 🤖 AI code generation IS recommended for this notebook.</small>

In [ ]:
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
import re
from transformers import Trainer, DataCollatorForLanguageModeling
import warnings

warnings.filterwarnings("ignore")

print("✅ Imports complete!")

- [ ] Download the necessary NLTK data for natural language processing:


In [ ]:
# Download NLTK data for tokenization and part-of-speech tagging
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

print("✅ NLTK data downloaded!")

## Context


### What is LoRA?

**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning technique that makes it possible to adapt large language models without modifying all their parameters.

Instead of updating the entire weight matrix during fine-tuning, LoRA:
1. Keeps the original model weights frozen
2. Introduces small, trainable "adapter" matrices
3. Learns a low-rank decomposition of the weight updates

This approach dramatically reduces:
- **Memory requirements**: Only ~0.1% of parameters need to be trained
- **Training time**: Fewer gradients to compute and update
- **Storage needs**: Save only the small adapter weights, not the entire model

### Our Task

We'll fine-tune DialoGPT (a conversational AI model) to automatically wrap nouns in special tokens. For example:
- Input: "I drink coffee in the morning"
- Output: "I drink noun{coffee} in the noun{morning}"

This task demonstrates how LoRA can teach models new formatting behaviors while preserving their conversational abilities.


## Step 1: Creating the Data Transformation Function


First, we need a function that can identify and wrap nouns in text. This will be used to create our training data.

- [ ] Create a function that uses NLTK's part-of-speech tagger to identify nouns:


In [ ]:
def wrap_nouns_in_text(text):
    """Transform text by wrapping nouns with noun{} tokens
    
    This function:
    1. Tokenizes the input text into words
    2. Tags each word with its part-of-speech
    3. Wraps nouns (NN tags) with noun{} markers
    """
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    wrapped_tokens = []
    for word, tag in tagged:
        # NN tags indicate nouns (NN, NNS, NNP, NNPS)
        if tag.startswith("NN"):
            wrapped_tokens.append(f"noun{{{word}}}")
        else:
            wrapped_tokens.append(word)

    return " ".join(wrapped_tokens)

# Test the function
test_text = "The cat drinks milk in the kitchen"
print(f"Original: {test_text}")
print(f"Wrapped:  {wrap_nouns_in_text(test_text)}")

## Step 2: Creating the Training Dataset


Now we'll create conversational examples where the model learns to respond with noun-wrapped text.

- [ ] Define conversation pairs and create a training dataset:


In [ ]:
def create_training_dataset():
    """Generate conversational dataset with noun wrapping
    
    Creates prompt-completion pairs where completions have wrapped nouns.
    Uses the DialoGPT format with <|endoftext|> tokens as separators.
    """
    conversations = [
        (
            "Tell me about your morning routine",
            "I wake up in my noun{bed}, brush my noun{teeth}, and have noun{coffee} while reading the noun{news}.",
        ),
        (
            "What's your favorite hobby?",
            "I enjoy reading noun{books} in the noun{library} and taking noun{walks} in the noun{park}.",
        ),
        (
            "Describe a typical workday",
            "I start by checking noun{emails} at my noun{desk}, attend noun{meetings} with noun{colleagues}, and work on noun{projects}.",
        ),
        (
            "What did you eat for lunch?",
            "I had a noun{sandwich} with fresh noun{vegetables} and a noun{glass} of cold noun{water}.",
        ),
        (
            "How do you relax?",
            "I listen to noun{music}, watch noun{movies}, or spend noun{time} with noun{friends} and noun{family}.",
        ),
        (
            "What's the weather like?",
            "The noun{sun} is shining brightly, with clear noun{skies} and a gentle noun{breeze}.",
        ),
        (
            "Tell me about your weekend plans",
            "I plan to visit the noun{museum}, have noun{dinner} at a nice noun{restaurant}, and read a new noun{book}.",
        ),
        (
            "Describe your home",
            "My noun{house} has a large noun{kitchen}, comfortable noun{living room}, and a peaceful noun{garden}.",
        ),
        (
            "What did you have for breakfast?",
            "I had some noun{eggs} with noun{toast} and drank noun{orange juice}.",
        ),
        (
            "Tell me about your workspace.",
            "My noun{office} has a big noun{desk}, comfortable noun{chair}, and several noun{monitors}.",
        ),
        (
            "How do you spend weekends?",
            "I like to visit noun{parks}, read noun{books}, and spend noun{time} with my noun{family}.",
        ),
    ]

    training_data = []
    # Create 300 training examples by cycling through our conversation pairs
    for i in range(300):
        prompt, completion = conversations[i % len(conversations)]
        # Format for DialoGPT with explicit conversation structure
        formatted_text = f"{prompt}<|endoftext|>{completion}<|endoftext|>"
        training_data.append({"text": formatted_text})

    return training_data

# Create and preview the dataset
dataset = create_training_dataset()
print(f"Created {len(dataset)} training examples")
print(f"\nExample entry:")
print(dataset[0]['text'][:200] + "...")

## Step 3: Setting Up the Model and Tokenizer


We'll use Microsoft's DialoGPT-medium model as our base. This model is specifically designed for conversational AI.

- [ ] Load the model with optional quantization for memory efficiency:


In [ ]:
def setup_model_and_tokenizer():
    """Load model and tokenizer with optimizations
    
    Features:
    - 4-bit quantization on GPU (if available) to reduce memory usage
    - Automatic device mapping
    - Proper padding token configuration
    """
    model_name = "microsoft/DialoGPT-medium"

    # Configure quantization for GPU if available
    bnb_config = None
    if torch.cuda.is_available():
        print("🎮 GPU detected! Using 4-bit quantization for efficiency")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    else:
        print("💻 Using CPU - this will be slower but still works!")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True,
    )

    return model, tokenizer

print("Loading model and tokenizer...")
model, tokenizer = setup_model_and_tokenizer()
print("✅ Model loaded successfully!")

## Step 4: Configuring LoRA


Now we'll configure LoRA to add trainable adapters to specific layers of the model.

Key LoRA parameters:
- **r (rank)**: Size of the low-rank matrices (lower = more efficient, higher = more expressive)
- **lora_alpha**: Scaling factor for the LoRA weights
- **target_modules**: Which model layers to add adapters to
- **lora_dropout**: Regularization to prevent overfitting

- [ ] Set up LoRA configuration and apply it to the model:


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank - lower values = fewer parameters to train
    lora_alpha=32,  # Scaling parameter
    target_modules=[
        "c_attn",  # Attention layers (query, key, value projections)
        "c_proj",  # Output projections
        "c_fc",    # Feed-forward layers
    ],
    lora_dropout=0.1,  # Dropout for regularization
    bias="none",  # Don't train biases
    task_type=TaskType.CAUSAL_LM,  # Language modeling task
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters info
trainable_params = 0
all_param = 0
for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Trainable params: {trainable_params:,} || All params: {all_param:,}")
print(f"Trainable %: {100 * trainable_params / all_param:.2f}%")

You should see that only a tiny fraction (<1%) of parameters are being trained!


## Step 5: Preparing Data for Training


- [ ] Convert our dataset to Hugging Face Dataset format and split it:


In [ ]:
# Create dataset splits
dataset = create_training_dataset()
train_dataset = Dataset.from_list(dataset[:270])  # 90% for training
eval_dataset = Dataset.from_list(dataset[270:])   # 10% for evaluation

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

- [ ] Tokenize the datasets:


In [ ]:
def tokenize_function(example):
    """Convert text to tokens for model input"""
    tokenized = tokenizer(
        example["text"], 
        truncation=True, 
        max_length=512,
        padding=False,  # Let data collator handle padding
        return_tensors=None  # Return lists, not tensors
    )
    return tokenized

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, remove_columns=["text"])

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM
)

print("✅ Data preparation complete!")

## Step 6: Training Configuration


- [ ] Set up training arguments that control the training process:


In [ ]:
training_args = TrainingArguments(
    output_dir="./noun_wrapper_lora",
    num_train_epochs=5,  # Number of full passes through the data
    per_device_train_batch_size=2,  # Small batch size for limited memory
    learning_rate=5e-4,  # Higher than typical for strong adaptation
    weight_decay=0.01,  # L2 regularization
    warmup_steps=10,  # Gradual learning rate increase
    logging_steps=5,
    save_steps=25,
    eval_steps=25,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    report_to=None,  # Disable experiment tracking
    remove_unused_columns=False,
)

print("✅ Training configuration set!")

## Step 7: Fine-Tuning with LoRA


- [ ] Create a Trainer and start the fine-tuning process:


In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("🚀 Starting LoRA fine-tuning...")
print("This may take several minutes depending on your hardware.")

# Train the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("./noun_wrapper_lora")
tokenizer.save_pretrained("./noun_wrapper_lora")

print("\n✅ Training complete! Model saved to ./noun_wrapper_lora")

## Step 8: Loading the Fine-Tuned Model


- [ ] Create functions to load and use the fine-tuned model:


In [ ]:
def load_trained_model():
    """Load the fine-tuned model with LoRA adapters"""
    print("Loading fine-tuned model...")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained("./noun_wrapper_lora")

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/DialoGPT-medium",
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    # Load LoRA adapters
    model = PeftModel.from_pretrained(base_model, "./noun_wrapper_lora")
    model.eval()  # Set to evaluation mode

    return model, tokenizer

# Load the model
model_finetuned, tokenizer_finetuned = load_trained_model()
print("✅ Model loaded!")

## Step 9: Generating Responses


- [ ] Create a function to generate responses with the fine-tuned model:


In [ ]:
def generate_response(model, tokenizer, prompt):
    """Generate response with the trained model
    
    Uses sampling strategies to reduce repetition:
    - Temperature: Controls randomness
    - Top-p (nucleus sampling): Limits token choices
    - Repetition penalty: Reduces repeated phrases
    """
    inputs = tokenizer(prompt + tokenizer.eos_token, return_tensors="pt").to(
        model.device
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,  # Limit response length
            do_sample=True,  # Enable sampling
            temperature=0.8,  # Slight randomness
            top_p=0.9,  # Nucleus sampling
            repetition_penalty=1.1,  # Discourage repetition
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated part (not the prompt)
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return response.strip()

print("✅ Response generation function ready!")

## Step 10: Testing the Fine-Tuned Model


- [ ] Test the model with various prompts to see if it learned to wrap nouns:


In [ ]:
test_prompts = [
    "What did you have for breakfast?",
    "Tell me about your workspace.",
    "How do you spend weekends?",
]

print("🧪 Testing the fine-tuned model:\n")
print("=" * 50)

for prompt in test_prompts:
    print(f"\n👤 User: {prompt}")
    response = generate_response(model_finetuned, tokenizer_finetuned, prompt)
    print(f"🤖 Assistant: {response}")
    
    # Count wrapped nouns
    noun_count = len(re.findall(r"noun\{[^}]+\}", response))
    print(f"📊 Wrapped nouns found: {noun_count}")
    print("-" * 50)

## Interactive Testing


- [ ] Try your own prompts to see how the model performs:


In [ ]:
# Interactive testing - modify the prompt to experiment!
custom_prompt = "What's your favorite food?"  # <-- Change this!

print(f"👤 User: {custom_prompt}")
response = generate_response(model_finetuned, tokenizer_finetuned, custom_prompt)
print(f"🤖 Assistant: {response}")

# Analyze the response
nouns_found = re.findall(r"noun\{([^}]+)\}", response)
if nouns_found:
    print(f"\n📝 Wrapped nouns: {', '.join(nouns_found)}")
else:
    print("\n⚠️ No wrapped nouns found - try more training or different prompts!")

## You did it!


You've successfully fine-tuned a language model using LoRA! This technique is at the heart of modern AI customization, enabling:
- Personal AI assistants adapted to specific writing styles
- Domain-specific chatbots for customer service
- Code completion models for proprietary codebases
- Medical or legal AI systems with specialized knowledge


## Key Takeaways

**LoRA is incredibly efficient.** Training <1% of parameters achieves task-specific adaptation while preserving the model's general capabilities. Full fine-tuning would require 100x more memory and compute.

**Small datasets work.** With just 300 examples, we taught the model a new formatting behavior. Traditional fine-tuning would require thousands of examples to avoid catastrophic forgetting.

**Adapters are modular.** LoRA weights can be swapped in/out, allowing one base model to serve multiple specialized tasks. Think of them as "skill plugins" for AI.

**Quantization enables accessibility.** 4-bit quantization makes large models runnable on consumer GPUs, democratizing AI customization.

## Real World Applications

- **Enterprise chatbots**: Adapt GPT models to company-specific terminology and policies
- **Code assistants**: Fine-tune on internal codebases while preserving general programming knowledge
- **Creative writing**: Train models to mimic specific authors' styles or genres
- **Scientific research**: Adapt models for domain-specific tasks (protein folding, drug discovery)
- **Multilingual support**: Fine-tune for low-resource languages without losing other language capabilities

## Follow-up Resources

- [PEFT Documentation](https://huggingface.co/docs/peft) - Complete guide to parameter-efficient methods
- [LoRA Paper](https://arxiv.org/abs/2106.09685) - Original research introducing the technique
- [QLoRA](https://arxiv.org/abs/2305.14314) - Combining LoRA with quantization for even better efficiency
- [Hugging Face Fine-tuning Guide](https://huggingface.co/docs/transformers/training) - Comprehensive training tutorials
- [LLM Fine-tuning Best Practices](https://github.com/huggingface/peft/tree/main/examples) - Production-ready examples

## Bonus: Quick Reference


Here's how to use your fine-tuned model in future projects:

```python
# Load the model
model, tokenizer = load_trained_model()

# Generate a response
response = generate_response(model, tokenizer, "Your prompt here")
print(response)
```

The model is saved in `./noun_wrapper_lora` and can be loaded anytime!